<a href="https://colab.research.google.com/github/tzs69/BT4012-kaggle-comp/blob/main/BT4012_competition_baseline_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Modelling (Traditional Models)
EDA conclusions
- Train: timesteps 1–35
- Test: timesteps 36–49
- Temporal CV required
- Positive label ≈ 11.7%
- Strong feature redundancy
- 59 PCs explain 95% variance
- Severe drift in feat_100/101/103 etc.
- Graph is snapshot-based
- Degree has label signal
- Strong graph homophily

To test out traditional ML classifiers

Models to test:
- Logistic Regression
- XGBoost
- Random Forest
- LightGBM
- CatBoost

In [ ]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 44.5 MB/s  0:00:02


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
# from google.colab import drive
from pathlib import Path
from importlib.metadata import version

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from catboost import CatBoostClassifier
from joblib import dump, load


from types import SimpleNamespace

In [ ]:
# drive.mount('/content/drive')
# ROOT_DIR = "/content/drive/MyDrive/BT4012_competition"

ROOT_DIR = "."

for folder in [
    "transformed_datasets",
    "base_models_cache",
    "submissions",
]:
    Path(ROOT_DIR + f"/{folder}").mkdir(parents=True, exist_ok=True)

df_train = pd.read_csv(ROOT_DIR + "/transformed_datasets/train_topo.csv")
df_test = pd.read_csv(ROOT_DIR + "/transformed_datasets/test_topo.csv")

labeled = df_train.loc[df_train["label"].notna()].reset_index(drop=True)
y = labeled["label"].astype(int)

raw_cols = [c for c in labeled.columns if c.startswith("feat_")]

counts_1hop_cols = ["1hop_in_count", "1hop_out_count"]
counts_2hop_cols = ["2hop_in_count", "2hop_out_count"]
all_counts_cols = counts_1hop_cols + counts_2hop_cols

onehop_cols = [
    f"1hop_{direction}_{stat}_{degree}"
    for direction in ("in", "out")
    for stat in ("mean", "max")
    for degree in ("in", "out")
]

twohop_cols = [
    f"2hop_{direction}_{stat}_{degree}"
    for direction in ("in", "out")
    for stat in ("mean", "max")
    for degree in ("in", "out")
]

feature_sets = {
    "raw": raw_cols,
    "counts_1hop": raw_cols + counts_1hop_cols,
    "counts_2hop": raw_cols + counts_2hop_cols,
    "counts_all": raw_cols + all_counts_cols,
    "onehop": raw_cols + counts_1hop_cols + onehop_cols,
    "twohop": raw_cols + counts_2hop_cols + twohop_cols,
    "all_topo": raw_cols + all_counts_cols + onehop_cols + twohop_cols,
}

required_cols = feature_sets["all_topo"]
assert set(required_cols).issubset(labeled.columns)
assert set(required_cols).issubset(df_test.columns)
assert labeled["txId"].is_unique
assert df_test["txId"].is_unique

Mounted at /content/drive


## Reusable Data

### CV Temporal Splits

In [ ]:
folds = [
    (range(1, 21), range(21, 26)),
    (range(1, 26), range(26, 31)),
    (range(1, 31), range(31, 36)),
]

cv_splits = []

for train_times, val_times in folds:
    train_idx = np.where(labeled["time_step"].isin(train_times))[0]
    val_idx = np.where(labeled["time_step"].isin(val_times))[0]
    cv_splits.append((train_idx, val_idx))

### Temporal Fold Train/Val Splits

In [ ]:
fold_windows = [
    (20, 21, 25),
    (25, 26, 30),
    (30, 31, 35),
]

times = labeled["time_step"]

cv_splits = [
    (
        np.flatnonzero(times.le(train_end)),
        np.flatnonzero(times.between(val_start, val_end)),
    )
    for train_end, val_start, val_end in fold_windows
]

for tr, va in cv_splits:
    assert times.iloc[tr].max() < times.iloc[va].min()

## Helper Functions

### Grid Search Cross Validation

In [ ]:
def run_gscv(model, param_grid, X, y, splits):
    search = GridSearchCV(
        model,
        param_grid,
        cv=splits,
        scoring="roc_auc",
        n_jobs=1,
        return_train_score=False,
    )
    return search.fit(X, y)

### Temporal Folds Validation

In [ ]:
def evaluate_temporal(model, X, y, splits):
    assert X.index.equals(y.index)

    scores = []
    oof = pd.Series(np.nan, index=X.index, name="prediction")

    for fold, (tr, va) in enumerate(splits, start=1):
        assert np.intersect1d(tr, va).size == 0
        assert oof.iloc[va].isna().all()

        fitted = clone(model).fit(X.iloc[tr], y.iloc[tr])
        pred = fitted.predict_proba(X.iloc[va])[:, 1]
        oof.iloc[va] = pred

        scores.append({
            "fold": fold,
            "n_train": len(tr),
            "n_val": len(va),
            "auc": roc_auc_score(y.iloc[va], pred),
            "ap": average_precision_score(y.iloc[va], pred),
        })

    return pd.DataFrame(scores), oof


### Run Predictions

In [ ]:
def fit_predict(model, X, y, X_test):
    assert X.index.equals(y.index)
    assert X.columns.equals(X_test.columns)

    fitted = clone(model).fit(X, y)
    return fitted, fitted.predict_proba(X_test)[:, 1]

### Persist Run Data

In [ ]:
def save_base_cache(name, model, cols, scores, oof, test_pred):
    # Check actual fitted inputs, incl. each model inside a seed avg
    members = model.estimators_ if isinstance(model, VotingClassifier) else [model]
    for member in members:
        estimator = member.steps[-1][1] if isinstance(member, Pipeline) else member
        names = getattr(estimator, "feature_names_in_", None)
        if names is None:
            names = estimator.feature_names_
        assert list(names) == list(cols), "Feature columns do not match the fitted model"

    # Assign validation fold numbers; earlier train-only rows stay -1
    fold_ids = np.full(len(labeled), -1, dtype=int)
    for fold, (_, va) in enumerate(cv_splits, start=1):
        fold_ids[va] = fold

    # Save OOF preds with assoc. txn IDs / labels / folds for final modelling
    oof_table = labeled[["txId", "time_step"]].copy()
    oof_table["label"] = y.to_numpy()
    oof_table["fold"] = fold_ids
    oof_table["prediction"] = oof.loc[labeled.index].to_numpy()

    # Save test preds with assoc. txn IDs
    test_table = df_test[["txId", "time_step"]].copy()
    test_table["prediction"] = np.asarray(test_pred)

    bundle = {
        "model": model,
        "feature_cols": list(cols),
        "oof": oof_table,
        "test_predictions": test_table,
        "cv_scores": scores,
    }

    cache_dir = ROOT_DIR + "/base_models_cache"
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir + f"/{name}.joblib"
    dump(bundle, cache_path)
    print(f"Saved {name}: {len(cols)} features -> {cache_path}")

### Seed Avg

In [ ]:
def make_seedavg(make_model, seed_param, seeds=(42, 43, 44)):
    base_model = make_model()

    return VotingClassifier(
        estimators=[
            (
                f"seed_{seed}",
                clone(base_model).set_params(**{seed_param: seed})
            )
            for seed in seeds
        ],
        voting="soft",
        n_jobs=1,
    )

## Choose feature sets

In [ ]:
logreg_cols = feature_sets["raw"]
xgb_cols = feature_sets["raw"]
rf_cols = feature_sets["raw"]
catboost_cols = feature_sets["raw"]
lgbm_cols = feature_sets["raw"]

## Logistic Regression

### Define LogReg Model

In [ ]:
def make_logreg_base():
    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "logreg",
            LogisticRegression(
                solver="lbfgs",
                penalty="l2",
                max_iter=2000,
            )
        )
    ])

logreg_base = make_logreg_base()
display(logreg_base)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg', LogisticRegression(max_iter=2000))])

### LogReg GSCV

In [ ]:
logreg_param_grid = {
    "logreg__C": np.logspace(-3, 3, 7),
    "logreg__class_weight": [None, "balanced"]
}

logreg_search = run_gscv(
    make_logreg_base(),
    logreg_param_grid,
    labeled[logreg_cols],
    y,
    cv_splits,
)

print(logreg_search.best_params_)
print(logreg_search.best_score_)

Best params: {'logreg__C': np.float64(1000.0), 'logreg__class_weight': None}
Best CV AUC: 0.9538864385623979


### LogReg Params Tuning

In [ ]:
logreg_search = SimpleNamespace(
    best_params_={
        "logreg__C": 1000.0,
        "logreg__class_weight": None,
    }
)

logreg_model = make_logreg_base().set_params(
    **logreg_search.best_params_
)

### LogReg Temporal Folds Validation

In [ ]:
logreg_scores, logreg_oof = evaluate_temporal(
    logreg_model,
    labeled[logreg_cols],
    y,
    cv_splits,
)

display(logreg_scores)

mean_auc = logreg_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,18889,5311,0.907660,0.704537
1,2,24200,2705,0.974345,0.860894
2,3,26905,4330,0.967066,0.720164


Mean AUC: 0.9496905600675191


### LogReg Submission Checkpoint
| Score | Date | Time | Remarks |
|-------|------|------|---------|
| 0.91645 | 05/10/26 | 01:05 | No degrees
| 0.91457 | 05/10/26 | 14:27 | Raw degrees
| 0.91963 | 08/10/26 | 23:35 | Log degrees

In [ ]:
logreg_final, logreg_test_pred = fit_predict(
    logreg_model,
    labeled[logreg_cols],
    y,
    df_test[logreg_cols],
)

## XGBoost

### Define XGBoost Model

In [ ]:
def make_xgb_base():
    return Pipeline([(
        "xgb",
        XGBClassifier(
            objective='binary:logistic',
            tree_method='hist',
            eval_metric='auc',
            random_state=42,
            n_jobs=2,
            min_child_weight=1.0,
            gamma=0.0,
            reg_alpha=0.0,
            reg_lambda=1.0,
            scale_pos_weight=1.0,
        )
    )])

xgb_base = make_xgb_base()
display(xgb_base)

Pipeline(steps=[('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='auc',
                               feature_types=None, feature_weights=None,
                               gamma=0.0, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=1.0, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=2,
                               num_parallel_tree=None, ...))])

### XGBoost GSCV

In [ ]:
xgb_param_grid = {
    "xgb__n_estimators": [100, 200, 300],
    "xgb__max_depth": [3, 5, 7],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0]
}

xgb_search = run_gscv(
    make_xgb_base(),
    xgb_param_grid,
    labeled[xgb_cols],
    y,
    cv_splits,
)

print(xgb_search.best_params_)
print(xgb_search.best_score_)

Best params: {'xgb__colsample_bytree': 1.0, 'xgb__learning_rate': 0.05, 'xgb__max_depth': 7, 'xgb__n_estimators': 200, 'xgb__subsample': 1.0}
Best CV AUC: 0.9836531875674001


### XGBoost Params Tuning

In [ ]:
xgb_search = SimpleNamespace(
    best_params_={
        "xgb__colsample_bytree": 1.0,
        "xgb__learning_rate": 0.05,
        "xgb__max_depth": 7,
        "xgb__n_estimators": 200,
        "xgb__subsample": 1.0,
    }
)

xgb_model = make_xgb_base().set_params(
    **xgb_search.best_params_
)

### XGBoost Temporal Folds Validation

In [ ]:
xgb_scores, xgb_oof = evaluate_temporal(
    xgb_model,
    labeled[xgb_cols],
    y,
    cv_splits,
)

display(xgb_scores)
mean_auc = xgb_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,18889,5311,0.959041,0.876202
1,2,24200,2705,0.994148,0.983568
2,3,26905,4330,0.997770,0.989623


Mean AUC: 0.9836531875674001


### XGBoost Submission Checkpoint
| Score | Date | Time | Remarks |
|-------|------|------|---------|
| 0.94228 | 05/10/26 | 01:02 | No degrees, 051026 gscv params
| 0.94454 | 08/10/26 | 23:31 | Log degrees
| 0.94454 | 08/10/26 | 23:33 | Raw degrees
| 0.94194 | 11/10/26 | 23:33 | Raw degrees, 111026 gscv params

In [ ]:
xgb_final, xgb_test_pred = fit_predict(
    xgb_model,
    labeled[xgb_cols],
    y,
    df_test[xgb_cols],
)

save_base_cache(
    "xgb",
    xgb_final,
    xgb_cols,
    xgb_scores,
    xgb_oof,
    xgb_test_pred,
)

Saved xgb: 165 features -> /content/drive/MyDrive/BT4012_competition/base_models_cache/xgb.joblib


## LightGBM

### Define LightGBM Model

In [ ]:
def make_lgbm_base():
    return Pipeline([
        (
            "lgbm",
            LGBMClassifier(
                objective="binary",
                boosting_type="gbdt",
                learning_rate=0.1,
                n_estimators=100,
                num_leaves=17,
                max_depth=-1,
                min_child_samples=33,
                colsample_bytree=0.5,
                subsample=1.0,
                subsample_freq=0,
                reg_alpha=0.04,
                reg_lambda=1.2,
                scale_pos_weight=1.0,
                random_state=42,
                n_jobs=2,
                verbosity=-1,
            )
        )
    ])


lgbm_base = make_lgbm_base()
display(lgbm_base)

Pipeline(steps=[('lgbm',
                 LGBMClassifier(colsample_bytree=0.5, min_child_samples=33,
                                n_jobs=2, num_leaves=17, objective='binary',
                                random_state=42, reg_alpha=0.04, reg_lambda=1.2,
                                scale_pos_weight=1.0, verbosity=-1))])

### LightGBM GSCV

In [ ]:
lgbm_param_grid = {
    "lgbm__n_estimators": [100, 300, 750],
    "lgbm__num_leaves": [17, 31],
    "lgbm__min_child_samples": [20, 33],
    "lgbm__colsample_bytree": [0.5, 1.0],
    "lgbm__scale_pos_weight": [4.0, 8.0],
}

lgbm_search = run_gscv(
    make_lgbm_base(),
    lgbm_param_grid,
    labeled[lgbm_cols],
    y,
    cv_splits,
)

print(lgbm_search.best_params_)
print(lgbm_search.best_score_)

{'lgbm__colsample_bytree': 1.0, 'lgbm__min_child_samples': 33, 'lgbm__n_estimators': 100, 'lgbm__num_leaves': 17, 'lgbm__scale_pos_weight': 8.0}
0.9813663885944391


### LightGBM Params Tuning

In [ ]:
lgbm_search = SimpleNamespace(
    best_params_={
        "lgbm__colsample_bytree": 1.0,
        "lgbm__min_child_samples": 33,
        "lgbm__n_estimators": 100,
        "lgbm__num_leaves": 17,
        "lgbm__scale_pos_weight": 8.0,
    }
)


lgbm_model = make_lgbm_base().set_params(
    **lgbm_search.best_params_
)

### LightGBM Temporal Folds Validation

In [ ]:
lgbm_scores, lgbm_oof = evaluate_temporal(
    lgbm_model,
    labeled[lgbm_cols],
    y,
    cv_splits,
)

display(lgbm_scores)
mean_auc = lgbm_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,18889,5311,0.952244,0.870014
1,2,24200,2705,0.993359,0.980490
2,3,26905,4330,0.998496,0.993215


Mean AUC: 0.9813663885944391


### LightGBM Submission Checkpoint
| Score | Date | Time | Remarks |
|-------|------|------|---------|
| 0.94388 | 18/10/26 | 02:24 |  |

In [ ]:
lgbm_final, lgbm_test_pred = fit_predict(
    lgbm_model,
    labeled[lgbm_cols],
    y,
    df_test[lgbm_cols],
)

save_base_cache(
    "lgbm",
    lgbm_final,
    lgbm_cols,
    lgbm_scores,
    lgbm_oof,
    lgbm_test_pred,
)

Saved lgbm: 165 features -> /content/drive/MyDrive/BT4012_competition/base_models_cache/lgbm.joblib


## Random Forest

### Define RF Model

In [ ]:
def make_rf_base():
    return Pipeline([(
        "rf",
        RandomForestClassifier(
            n_estimators=150,
            random_state=42
        )
    )])

rf_base = make_rf_base()
display(rf_base)

Pipeline(steps=[('rf',
                 RandomForestClassifier(n_estimators=150, random_state=42))])

### RF GSCV

In [ ]:
rf_param_grid = {
    "rf__max_depth": [None, 10, 20],
    "rf__min_samples_leaf": [1, 3, 5],
    "rf__max_features": ["sqrt", 0.5],
    "rf__class_weight": [None, "balanced"]
}

rf_search = run_gscv(
    make_rf_base(),
    rf_param_grid,
    labeled[rf_cols],
    y,
    cv_splits,
)

print(rf_search.best_params_)
print(rf_search.best_score_)

### RF Params Tuning

In [ ]:
## Best params obtained from my last run gscv; didnt run again as its extremely time consuming
rf_search = SimpleNamespace(
    best_params_={
        "rf__max_depth": 20,
        "rf__min_samples_leaf": 1,
        "rf__max_features": "sqrt",
        "rf__class_weight": "balanced",
    }
)

rf_model = make_rf_base().set_params(
    **rf_search.best_params_
)

### RF Temporal Folds Validation

In [ ]:
rf_scores, rf_oof = evaluate_temporal(
    rf_model,
    labeled[rf_cols],
    y,
    cv_splits,
)

display(rf_scores)
mean_auc = rf_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,18889,5311,0.923621,0.846905
1,2,24200,2705,0.994664,0.984811
2,3,26905,4330,0.997382,0.991783


Mean AUC: 0.9718889221755406


### RF Submission Checkpoint
| Score | Date | Time | Remarks |
|-------|------|------|---------|
| 0.92930 | 05/10/26 | 14:24 | No degrees |
| 0.92661 | 05/10/26 | 14:24 | Raw degrees |

In [ ]:
rf_final, rf_test_pred = fit_predict(
    rf_model,
    labeled[rf_cols],
    y,
    df_test[rf_cols],
)

save_base_cache(
    "rf",
    rf_final,
    rf_cols,
    rf_scores,
    rf_oof,
    rf_test_pred,
)

Saved rf: 169 fitted inputs -> /content/drive/MyDrive/BT4012_competition/base_models_cache/rf.joblib


## CatBoost

### Define CatBoost Model

In [ ]:
def make_catboost_base():
    return Pipeline([(
        "catboost",
        CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=500,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            random_seed=42,
            thread_count=2,
            task_type="CPU",
            boosting_type="Plain",
            use_best_model=False,
            verbose=False,
            allow_writing_files=False,
        )
    )])

catboost_base = make_catboost_base()
display(catboost_base)


Pipeline(steps=[('catboost',
                 CatBoostClassifier(allow_writing_files=False, boosting_type='Plain', depth=6, eval_metric='AUC', iterations=500, l2_leaf_reg=3.0, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='CPU', thread_count=2, use_best_model=False, verbose=False))])

### CatBoost Temporal Folds Validation

In [ ]:
catboost_model = make_catboost_base()

catboost_model = make_seedavg(
    lambda: make_catboost_base(),
    seed_param="catboost__random_seed",
)

catboost_scores, catboost_oof = evaluate_temporal(
    catboost_model,
    labeled[catboost_cols],
    y,
    cv_splits,
)

display(catboost_scores)
mean_auc = catboost_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,18889,5311,0.898759,0.850627
1,2,24200,2705,0.994868,0.987200
2,3,26905,4330,0.998507,0.992028


Mean AUC: 0.9640446465797341


### CatBoost Submission Checkpoint

In [ ]:
catboost_final, catboost_test_pred = fit_predict(
    catboost_model,
    labeled[catboost_cols],
    y,
    df_test[catboost_cols],
)

save_base_cache(
    "catboost",
    catboost_final,
    catboost_cols,
    catboost_scores,
    catboost_oof,
    catboost_test_pred,
)

Saved catboost: 165 features -> /content/drive/MyDrive/BT4012_competition/base_models_cache/catboost.joblib
